# Export final trained artifacts for the local project

Run this notebook **after** Model A evaluation/export and Model B full-data fine-tuning/evaluation have completed. It copies only model weights, tokenizers, metrics, training history, and test predictions from Google Drive into one ZIP. It does **not** include the 5+ GB raw Yelp file, the multi-GB tokenized cache, or Kaggle credentials.

Expected Drive folders:
- `MyDrive/yelp_model_a/checkpoints_v2/` and `artifacts_v2/`
- `MyDrive/yelp_model_b/model/` and `artifacts/`

If `yelp_model_a` is shared by a teammate, add a Drive shortcut to that shared folder inside your own My Drive with the exact name `yelp_model_a` before running this export.

In [ ]:
from pathlib import Path
import hashlib, json, shutil, zipfile
from google.colab import drive, files

drive.mount("/content/drive")
MYDRIVE = Path("/content/drive/MyDrive")
A_ROOT = MYDRIVE / "yelp_model_a"
B_ROOT = MYDRIVE / "yelp_model_b"
BUNDLE_ROOT = Path("/content/NLP_Task3_Final_Artifacts")
if BUNDLE_ROOT.exists():
    shutil.rmtree(BUNDLE_ROOT)
(BUNDLE_ROOT / "artifacts/model_a/tokenizer").mkdir(parents=True, exist_ok=True)
(BUNDLE_ROOT / "artifacts/model_b/model").mkdir(parents=True, exist_ok=True)


In [ ]:
def copy_required(src: Path, dst: Path):
    if not src.is_file():
        raise FileNotFoundError(f"Required artifact is missing: {src}")
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print("COPIED", src, "->", dst.relative_to(BUNDLE_ROOT))

# Model A: checkpoint, measured metrics, training history, paired predictions, error analysis.
A_OUT = BUNDLE_ROOT / "artifacts/model_a"
copy_required(A_ROOT / "checkpoints_v2/best_model.pt", A_OUT / "best_model.pt")
for name in ["metrics.json", "training_history.csv", "error_analysis_top50.csv", "test_predictions.csv"]:
    copy_required(A_ROOT / "artifacts_v2" / name, A_OUT / name)

# Model A tokenizer files are saved beside the ONNX export by the Model A notebook.
A_TOK_SRC = A_ROOT / "artifacts_v2/onnx"
for name in ["tokenizer.json", "tokenizer_config.json", "special_tokens_map.json", "vocab.txt", "added_tokens.json"]:
    candidate = A_TOK_SRC / name
    if candidate.exists():
        copy_required(candidate, A_OUT / "tokenizer" / name)
if not (A_OUT / "tokenizer/vocab.txt").exists():
    raise FileNotFoundError("Model A tokenizer vocab.txt was not found in artifacts_v2/onnx")

# Model B: exported Hugging Face model/tokenizer and measured evaluation artifacts.
B_OUT = BUNDLE_ROOT / "artifacts/model_b"
B_MODEL_SRC = B_ROOT / "model"
if not (B_MODEL_SRC / "config.json").exists():
    raise FileNotFoundError(f"Model B exported model folder is missing: {B_MODEL_SRC}")
shutil.copytree(B_MODEL_SRC, B_OUT / "model", dirs_exist_ok=True)
for name in ["metrics.json", "training_history.csv", "classification_report.csv",
             "confusion_matrix.png", "test_predictions.csv", "error_analysis_top100.csv"]:
    candidate = B_ROOT / "artifacts" / name
    if candidate.exists():
        copy_required(candidate, B_OUT / name)
copy_required(B_ROOT / "artifacts/metrics.json", B_OUT / "metrics.json")
copy_required(B_ROOT / "artifacts/test_predictions.csv", B_OUT / "test_predictions.csv")
if (B_ROOT / "pretrained_inference.py").exists():
    copy_required(B_ROOT / "pretrained_inference.py", B_OUT / "pretrained_inference.py")


In [ ]:
# Write a checksum manifest so the local install can be audited.
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

manifest = {"bundle": "NLP Task 3 final model artifacts", "files": []}
for path in sorted((BUNDLE_ROOT / "artifacts").rglob("*")):
    if path.is_file():
        manifest["files"].append({"path": str(path.relative_to(BUNDLE_ROOT)),
                                  "size_bytes": path.stat().st_size, "sha256": sha256_file(path)})
(BUNDLE_ROOT / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
zip_path = Path("/content/NLP_Task3_Final_Artifacts.zip")
if zip_path.exists(): zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
    for path in sorted(BUNDLE_ROOT.rglob("*")):
        if path.is_file():
            zf.write(path, path.relative_to(BUNDLE_ROOT))
print("\nArtifact bundle ready:", zip_path)
print("Bundle size (MB):", round(zip_path.stat().st_size / 1024**2, 1))
print("Files included:", len(manifest["files"]))
files.download(str(zip_path))
